<a href="https://colab.research.google.com/github/sangeerthP-10/Sangeerth_P_Task_and_Assment_IN126008302_Innomatics/blob/main/IN126008302_GEN_AI/Task__02_Sentiment_Analysis_using_NLP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#1.Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

#2.Load Dataset(TWITTER DATA)

In [ ]:
df = pd.read_csv("Twitter_Data.csv")

df.head()

,clean_text,category
0,when modi promised “minimum government maximum...,-1.0
1,talk all the nonsense and continue all the dra...,0.0
2,what did just say vote for modi welcome bjp t...,1.0
3,asking his supporters prefix chowkidar their n...,1.0
4,answer who among these the most powerful world...,1.0


#3. Data Understanding

In [ ]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)

print("\nClass Distribution:")
print(df['category'].value_counts())

print("\nSample Text:")
print(df['clean_text'][0])

Shape: (162980, 2)

Columns: Index(['clean_text', 'category'], dtype='object')

Class Distribution:
category
 1.0    72250
 0.0    55213
-1.0    35510
Name: count, dtype: int64

Sample Text:
when modi promised “minimum government maximum governance” expected him begin the difficult job reforming the state why does take years get justice state should and not business and should exit psus and temples


In [ ]:
df = df.dropna()

#4.NLP Preprocessing

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [ ]:
def preprocess(text):
    text = text.lower()

    text = re.sub(r'http\S+', '', text)   # remove URLs
    text = re.sub(r'@\w+', '', text)      # remove mentions
    text = re.sub(r'#\w+', '', text)      # remove hashtags
    text = re.sub(r'[^a-zA-Z\s]', '', text)  # remove punctuation

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens)

In [ ]:
df['processed_text'] = df['clean_text'].apply(preprocess)

#5.Features & Labels

In [ ]:
X = df['processed_text']
y = df['category']

#6.Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

#7.Feature Engineering

**Bag of Words**

In [ ]:
bow = CountVectorizer(max_features=5000)

X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)

**TF-IDF**

In [ ]:
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

#8.Model Training

**Logistic Regression**

In [ ]:
lr = LogisticRegression(max_iter=200)

lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)

**Naive Bayes**

In [ ]:
nb = MultinomialNB()

nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)

**Decision Tree**

In [64]:
dt = DecisionTreeClassifier()

dt.fit(X_train_tfidf, y_train)
y_pred_dt = dt.predict(X_test_tfidf)

#9.Evaluation Function

In [65]:
def evaluate(name, y_test, y_pred):
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred, average='weighted'))
    print("Recall:", recall_score(y_test, y_pred, average='weighted'))
    print("F1 Score:", f1_score(y_test, y_pred, average='weighted'))

#10.Evaluate Models

In [66]:
evaluate("Logistic Regression", y_test, y_pred_lr)
evaluate("Naive Bayes", y_test, y_pred_nb)
evaluate("Decision Tree", y_test, y_pred_dt)


Logistic Regression
Accuracy: 0.8696692642817696
Precision: 0.8713722005744721
Recall: 0.8696692642817696
F1 Score: 0.8682007061690278

Naive Bayes
Accuracy: 0.7247652942259312
Precision: 0.7459216399429696
Recall: 0.7247652942259312
F1 Score: 0.7154553434392134

Decision Tree
Accuracy: 0.7967110511137019
Precision: 0.7954983020559523
Recall: 0.7967110511137019
F1 Score: 0.7960092302668117


#12.Comparison Table

In [ ]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "Decision Tree"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_dt)
    ],
    "F1 Score": [
        f1_score(y_test, y_pred_lr, average='weighted'),
        f1_score(y_test, y_pred_nb, average='weighted'),
        f1_score(y_test, y_pred_dt, average='weighted')
    ]
})

results

,Model,Accuracy,F1 Score
0,Logistic Regression,0.869669,0.868201
1,Naive Bayes,0.724765,0.715455
2,Decision Tree,0.798644,0.797929


## Final Insights

- Logistic Regression performed best due to its efficiency with sparse text features.  
- Naive Bayes is fast and performs well but assumes feature independence.  
- Decision Tree tends to overfit on high-dimensional text data.  

---

## Best Choices

- **Best Preprocessing:** Lemmatization + Stopword Removal  
- **Best Vectorization:** TF-IDF  
- **Best Model:** Logistic Regression  

---

## Trade-offs

| Model                | Advantage       | Disadvantage   |
|---------------------|----------------|----------------|
| Logistic Regression | High accuracy  | Slightly slower |
| Naive Bayes         | Fast           | Less flexible   |
| Decision Tree       | Interpretable  | Overfitting     |